In [31]:
import json
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
import os

def load_result():
    possible_paths = [
        "result.json",
        "outputs/result.json"
    ]
    for p in possible_paths:
        if os.path.exists(p):
            with open(p, encoding='utf-8') as f:
                return json.load(f)
    raise Exception("no finding result.json")

data = load_result()
summary = data.get("summary", {})
parsed_trades = data.get("parsed_trades", [])
compliance_results = data.get("compliance_results", [])
novel_trade_ids = summary.get("novel_instrument_trade_ids", [])

In [32]:
# Chart 1: Portfolio compliance heatmap
import plotly.express as px
import json

# --------------- 1. pull result.json ---------------
with open("result.json", "r", encoding="utf-8") as f:
    data = json.load(f)

parsed_trades = data.get("parsed_trades", [])
compliance_results = data.get("compliance_results", [])

compliance_map = {}
for item in compliance_results:
    tid = item["trade_id"]
    regime = item["regime"]
    status = item["status"]

    if tid not in compliance_map:
        compliance_map[tid] = {}
    compliance_map[tid][regime] = status

# --------------- 2. prepare data ---------------
trade_ids = [t["trade_id"] for t in parsed_trades]
regimes = ["CFTC", "EMIR"]

status_list = ["COMPLIANT", "CONDITIONAL", "NOT_APPLICABLE", "NONCOMPLIANT"]
color_list = ["#22c55e", "#eab308", "#94a3b8", "#ef4444"]
status_to_num = {s: i for i, s in enumerate(status_list)}

z_data = []
text_data = []
for tid in trade_ids:
    row_num = []
    row_text = []
    # CFTC
    cftc = compliance_map.get(tid, {}).get("CFTC", "NONCOMPLIANT")
    row_num.append(status_to_num[cftc])
    row_text.append(cftc)
    # EMIR
    emir = compliance_map.get(tid, {}).get("EMIR", "NONCOMPLIANT")
    row_num.append(status_to_num[emir])
    row_text.append(emir)

    z_data.append(row_num)
    text_data.append(row_text)

# --------------- 3. draw heatmap ---------------
print('1. Portfolio compliance heatmap')
fig = px.imshow(
    z_data,
    x=regimes,
    y=trade_ids,
    color_continuous_scale=color_list,
    range_color=[0, 3],
    title="Portfolio Compliance Heatmap",
    height=800,
    labels=dict(color="Status"),
    aspect="auto"  # 让单元格比例更合理
)

# --------------- 4. correct text ---------------
fig.update_traces(
    text=text_data,
    texttemplate="%{text}",
    textfont={"size": 10, "color": "white"},
    hovertemplate="Trade: %{y}<br>Regime: %{x}<br>Status: %{text}<extra></extra>"
)

# --------------- 5. correct colour---------------
fig.update_layout(
    coloraxis_colorbar=dict(
        tickvals=[0, 1, 2, 3],
        ticktext=status_list,
        title="Status",
        title_side="right"
    ),
    plot_bgcolor="#121212",
    paper_bgcolor="#121212",
    font_color="white",
    xaxis_title="Regime",
    yaxis_title="Trade ID",
    title_x=0.5
)

fig.show()

1. Portfolio compliance heatmap


In [33]:
# Chart 2: Error frequency chart
print("2. Error frequency chart")

errors = []
for c in compliance_results:
    for f in c.get("findings", []):
        errors.append(f.get("field", "unknown"))

if errors:
    df_err = pd.Series(errors).value_counts().reset_index()
    df_err.columns = ["field", "count"]
    fig2 = px.bar(df_err, x="count", y="field", orientation="h",
                   title="Error Field Frequency",
                   color="count", color_continuous_scale="Reds")
    fig2.show()
else:
    print("no error data")

2. Error frequency chart


In [34]:
# Chart 3：Asset class breakdown
print("3. Asset class breakdown")

cls = summary.get("classification_counts", {})
labels = []
values = []
for key, val in cls.items():
    if key == "CONVENTIONAL_DERIVATIVE":
        labels.append("Conventional Derivatives")
    elif key == "NOVEL_INSTRUMENT_NO_TAXONOMY":
        labels.append("Novel / Unclassified Instruments")
    else:
        labels.append(key)
    values.append(val)

colors = ["#6366f1", "#f87171"]

fig3 = px.pie(
    names=labels,
    values=values,
    title="Asset Class Breakdown",
    color_discrete_sequence=colors
)

fig3.update_traces(
    textinfo="percent+label",
    textposition="outside",
    textfont={"size": 14},
    insidetextorientation="horizontal"
)

fig3.show()

3. Asset class breakdown


In [35]:
import pandas as pd
from IPython.display import display, HTML

print("4. Classification frontier panel")

table_data = []
for trade_id in novel_trade_ids:
    cftc_status = "N/A"
    emir_status = "N/A"
    cftc_notes = []
    emir_notes = []

    for c in compliance_results:
        if c["trade_id"] == trade_id:
            if c["regime"] == "CFTC":
                cftc_status = c["status"]
                for f in c.get("findings", []):
                    cftc_notes.append(f["message"])
            elif c["regime"] == "EMIR":
                emir_status = c["status"]
                for f in c.get("findings", []):
                    emir_notes.append(f["message"])

    table_data.append({
        "Trade ID": trade_id,
        "CFTC Status": cftc_status,
        "CFTC Compliance Notes": "\n".join(cftc_notes) if cftc_notes else "No issues",
        "EMIR Status": emir_status,
        "EMIR Compliance Notes": "\n".join(emir_notes) if emir_notes else "No issues"
    })

df_table = pd.DataFrame(table_data)

def highlight_status(val):
    if val == "PASS":
        return "background-color: #4ade80; color: white;"
    elif val == "CONDITIONAL":
        return "background-color: #fbbf24; color: #1f2937;"
    elif val == "NOT_APPLICABLE":
        return "background-color: #9ca3af; color: white;"
    else:
        return "background-color: #f87171; color: white;"

styled_table = df_table.style.map(
    highlight_status, subset=["CFTC Status", "EMIR Status"]
).hide(axis='index') \
.set_properties(**{
    'white-space': 'pre-wrap',
    'border': '1px solid #374151',
    'padding': '10px',
    'color': '#e5e7eb',
    'text-align': 'left'
}).set_table_styles([{
    'selector': 'th',
    'props': [
        ('background-color', '#1f2937'),
        ('color', '#f9fafb'),
        ('text-align', 'left'),
        ('padding', '12px')
    ]
}]).set_caption("T026-T028 Jurisdictional Asymmetry Compliance Table")

display(styled_table)

4. Classification frontier panel


Trade ID,CFTC Status,CFTC Compliance Notes,EMIR Status,EMIR Compliance Notes
T026,CONDITIONAL,EventContract on a CFTC-regulated DCM is treated as conditional scope pending classification.,NOT_APPLICABLE,EventContract is outside the EMIR OTC derivative reporting taxonomy in this project.
T027,NOT_APPLICABLE,EventContract is not traded on a CFTC-regulated DCM and is outside CFTC OTC reporting for this project.,NOT_APPLICABLE,EventContract is outside the EMIR OTC derivative reporting taxonomy in this project.
T028,CONDITIONAL,EventContract on a CFTC-regulated DCM is treated as conditional scope pending classification.,NOT_APPLICABLE,EventContract is outside the EMIR OTC derivative reporting taxonomy in this project.
